[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day27_cnns/day27_notebook.ipynb)

# Day 27 / 42: Convolutional Neural Networks (CNNs)
### #42DaysOfML | Week 4: Deep Learning

---

## What You'll Learn
- What convolution actually does to an image
- How CNNs learn edges → shapes → objects in layers
- Pooling, padding, and stride
- Build and train a CNN on CIFAR-10 with PyTorch
- Visualise what the first conv layer actually learns
- Real production problem: model accuracy vs inference speed tradeoff

---

In [ ]:
# Run this cell first — installs everything needed
!pip install torch torchvision matplotlib numpy --quiet

## The Concept

A regular neural network (fully connected) treats every pixel as an independent input. For a 224×224 RGB image, that's 150,528 inputs. It has no idea that the pixel at position (10, 10) is spatially related to the pixel at (11, 10). It also needs to relearn "what an edge looks like" for every possible location in the image.

CNNs fix this with **convolution**: sliding a small filter (kernel) over the image and computing dot products at each position. The same filter scans the entire image, so if it learns to detect a vertical edge at position (10, 10), it can detect the same edge anywhere in the image. This is called **weight sharing** and it's what makes CNNs efficient.

**The three key operations:**
- **Convolution**: Apply a learnable filter over the input. One filter = one feature map.
- **Pooling**: Downsample the feature map (max pooling keeps the strongest activation in a region). Reduces spatial size and computation.
- **Activation (ReLU)**: Adds non-linearity after each conv layer.

**How CNN layers build up understanding:**
- Layer 1: Detects edges and color gradients
- Layer 2: Combines edges into corners and textures
- Layer 3+: Combines textures into object parts (wheels, eyes, ears)
- Final layers: Combines parts into full objects

This is not a theory — it's been directly visualised in papers. Zeiler and Fergus (2014) published visualisations of AlexNet filters that showed exactly this hierarchy.

**Key terms:**
- **Kernel/Filter**: Small matrix (e.g., 3×3) that slides over input. Contains learnable weights.
- **Feature Map**: Output of applying one filter to the input.
- **Stride**: How many pixels the filter moves per step. Stride=2 halves the output size.
- **Padding**: Adding zeros around the input border so the output stays the same size as the input ("same" padding).
- **Channels**: Depth of the input. RGB image = 3 channels. After a conv layer with 32 filters, output has 32 channels.

In [ ]:
# ============================================================
# SECTION 1: Understand Convolution From Scratch
# Build convolution manually with NumPy before using PyTorch
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Simulate a simple 8x8 grayscale image with a clear edge
image = np.array([
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 1, 1, 1, 1],
], dtype=float)

# Vertical edge detection filter (Sobel-style)
# Positive left, negative right -> activates at vertical transitions
vertical_edge_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

# Horizontal edge detection filter
horizontal_edge_filter = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
], dtype=float)

def manual_conv2d(image, kernel, stride=1):
    """Manual 2D convolution — same math PyTorch/TensorFlow use internally."""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(0, out_h):
        for j in range(0, out_w):
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(region * kernel)
    return output

v_feature_map = manual_conv2d(image, vertical_edge_filter)
h_feature_map = manual_conv2d(image, horizontal_edge_filter)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Original Image\n(vertical edge at col 4)', fontsize=10)
axes[1].imshow(vertical_edge_filter, cmap='RdBu'); axes[1].set_title('Vertical Edge Filter\n(3x3 kernel)', fontsize=10)
axes[2].imshow(v_feature_map, cmap='RdBu'); axes[2].set_title(f'Feature Map (Vertical)\nOutput size: {v_feature_map.shape}', fontsize=10)
axes[3].imshow(h_feature_map, cmap='RdBu'); axes[3].set_title(f'Feature Map (Horizontal)\nOutput size: {h_feature_map.shape}', fontsize=10)

for ax in axes:
    ax.axis('off')

plt.suptitle('Manual Convolution: Input → Filter → Feature Map', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Input shape:          {image.shape}")
print(f"Filter shape:         {vertical_edge_filter.shape}")
print(f"Output shape:         {v_feature_map.shape}")
print(f"\nVertical edge filter activates strongly at column 3-4 (the actual edge):")
print(v_feature_map)

In [ ]:
# ============================================================
# SECTION 2: Stride and Padding — What They Do to Output Size
# ============================================================

def compute_output_size(input_size, kernel_size, stride, padding):
    return (input_size + 2*padding - kernel_size) // stride + 1

print("OUTPUT SIZE FORMULA: (input_size + 2*padding - kernel_size) // stride + 1")
print("=" * 60)

configs = [
    (8, 3, 1, 0, "No padding, stride 1 (default)"),
    (8, 3, 1, 1, "Same padding, stride 1 (output = input size)"),
    (8, 3, 2, 0, "No padding, stride 2 (downsamples by ~2x)"),
    (8, 3, 2, 1, "Same padding, stride 2 (cleaner downsampling)"),
]

for inp, k, s, p, desc in configs:
    out = compute_output_size(inp, k, s, p)
    print(f"{desc}")
    print(f"  Input: {inp}x{inp} | Kernel: {k}x{k} | Stride: {s} | Padding: {p} -> Output: {out}x{out}")
    print()

print("KEY INSIGHT:")
print("- Stride=1, Padding=1 with 3x3 kernel: output size = input size ('same' padding)")
print("- Stride=2: halves spatial dimensions (like MaxPool but learnable)")
print("- No padding: each layer slightly shrinks the feature map")

In [ ]:
# ============================================================
# SECTION 3: Max Pooling — Why and How
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

feature_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 1, 7, 8],
    [2, 4, 6, 5]
], dtype=float)

def max_pool2d(feature_map, pool_size=2, stride=2):
    """Max pooling: take the maximum value in each pool window."""
    h, w = feature_map.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = feature_map[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)
    return output

pooled = max_pool2d(feature_map)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im1 = axes[0].imshow(feature_map, cmap='Blues')
axes[0].set_title(f'Feature Map: {feature_map.shape}', fontsize=12)
for i in range(feature_map.shape[0]):
    for j in range(feature_map.shape[1]):
        axes[0].text(j, i, int(feature_map[i,j]), ha='center', va='center', fontsize=14, fontweight='bold')

# Draw pool boundaries
for i in [0, 2, 4]:
    axes[0].axhline(i - 0.5, color='red', linewidth=2)
    axes[0].axvline(i - 0.5, color='red', linewidth=2)

im2 = axes[1].imshow(pooled, cmap='Greens')
axes[1].set_title(f'After Max Pool 2x2: {pooled.shape}', fontsize=12)
for i in range(pooled.shape[0]):
    for j in range(pooled.shape[1]):
        axes[1].text(j, i, int(pooled[i,j]), ha='center', va='center', fontsize=14, fontweight='bold')

plt.suptitle('Max Pooling: Keeps the strongest activation in each 2x2 region', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Original feature map: {feature_map.shape} -> After Max Pool 2x2: {pooled.shape}")
print("Spatial size halved. Most active features preserved.")
print("This gives CNNs translation invariance: the cat is a cat whether it's top-left or bottom-right.")

In [ ]:
# ============================================================
# SECTION 4: Build and Train a CNN on CIFAR-10 with PyTorch
# CIFAR-10: 60,000 images, 10 classes, 32x32 RGB
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    # Normalize: mean and std for CIFAR-10 (precomputed)
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Downloads ~170MB to ./data/ — only happens once
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training samples:   {len(trainset):,}")
print(f"Test samples:       {len(testset):,}")
print(f"Image size:         32x32x3 (RGB)")
print(f"Classes:            {classes}")
print(f"Batches per epoch:  {len(trainloader)}")

In [ ]:
# Visualise sample images
def imshow(img):
    # Unnormalise
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    img = img * std + mean
    img = torch.clamp(img, 0, 1)
    return img.permute(1, 2, 0).numpy()

dataiter = iter(trainloader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imshow(images[i]))
    ax.set_title(classes[labels[i]], fontsize=9)
    ax.axis('off')

plt.suptitle('Sample CIFAR-10 Images (32x32 RGB)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Define the CNN Architecture
# ============================================================

class CNNClassifier(nn.Module):
    """
    A CNN for CIFAR-10 image classification.
    
    Architecture:
      Input: 3x32x32 (RGB image)
      Conv Block 1: 32 filters, 3x3
      Conv Block 2: 64 filters, 3x3
      Conv Block 3: 128 filters, 3x3
      Classifier:   FC layers -> 10 classes
    """
    def __init__(self):
        super(CNNClassifier, self).__init__()
        
        # Block 1: learns basic edges and color patterns
        # Input: 3x32x32 -> Output: 32x16x16
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32 -> 32x16x16
            nn.Dropout2d(0.25)
        )
        
        # Block 2: learns textures and simple shapes
        # Input: 32x16x16 -> Output: 64x8x8
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16 -> 8x8
            nn.Dropout2d(0.25)
        )
        
        # Block 3: learns object parts
        # Input: 64x8x8 -> Output: 128x4x4
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 8x8 -> 4x4
            nn.Dropout2d(0.25)
        )
        
        # Classifier: takes flattened features and outputs 10 class scores
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),  # 128 channels x 4x4 spatial = 2048 features
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 10)  # 10 CIFAR-10 classes
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = CNNClassifier().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Architecture:")
print(model)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ============================================================
# Train the CNN
# ~10 epochs gets to ~75% accuracy on CIFAR-10
# With GPU: ~5 minutes. CPU: ~30 minutes.
# ============================================================
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Learning rate scheduler: reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

NUM_EPOCHS = 10
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(NUM_EPOCHS):
    start = time.time()
    
    # --- Training phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()      # clear previous gradients
        outputs = model(inputs)    # forward pass
        loss = criterion(outputs, targets)  # compute loss
        loss.backward()            # backpropagation
        optimizer.step()           # update weights
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    train_loss = running_loss / len(trainloader)
    train_acc = 100. * correct / total
    
    # --- Validation phase ---
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()
    
    val_loss = val_loss_sum / len(testloader)
    val_acc = 100. * val_correct / val_total
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    elapsed = time.time() - start
    print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.1f}% | "
          f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.1f}% | "
          f"Time: {elapsed:.1f}s")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss', color='#2196F3', linewidth=2)
ax1.plot(val_losses, label='Val Loss', color='#F44336', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train Accuracy', color='#2196F3', linewidth=2)
ax2.plot(val_accs, label='Val Accuracy', color='#F44336', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training vs Validation Accuracy', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy: {val_accs[-1]:.1f}%")
print(f"(Random baseline: 10% | A good CNN on CIFAR-10: 75-85% in 10 epochs)")

In [ ]:
# ============================================================
# SECTION 5: Visualise What the First Conv Layer Learns
# This is what Zeiler & Fergus did in their 2014 paper
# ============================================================

# Get the weights of the first conv layer: 32 filters, each 3x3x3 (RGB)
first_conv_weights = model.block1[0].weight.data.cpu()

print(f"First conv layer weights shape: {first_conv_weights.shape}")
print(f"-> 32 filters, each looking at 3 channels (RGB) with a 3x3 spatial kernel")

# Normalize each filter for display
def normalize_filter(f):
    f = f - f.min()
    if f.max() > 0:
        f = f / f.max()
    return f

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    filt = first_conv_weights[i]  # Shape: 3x3x3
    filt = filt.permute(1, 2, 0)  # -> 3x3x3 for RGB display
    filt = normalize_filter(filt.numpy())
    ax.imshow(filt)
    ax.set_title(f'F{i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('First Conv Layer: 32 Learned Filters (3x3 RGB)\n'
             'These detect basic edges, gradients, and color patterns', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nWhat you're seeing:")
print("- Each 3x3 patch is a learned filter from training on 50,000 CIFAR-10 images")
print("- Filters with strong direction = edge detectors (horizontal, vertical, diagonal)")
print("- Filters with solid color = color blob detectors")
print("- Deeper layers learn combinations of these -> shapes -> object parts -> objects")

In [ ]:
# ============================================================
# SECTION 6: Per-Class Accuracy Analysis
# ============================================================

class_correct = [0] * 10
class_total = [0] * 10

model.eval()
with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        for t, p in zip(targets, predicted):
            class_correct[t.item()] += (t == p).item()
            class_total[t.item()] += 1

per_class_acc = [100 * class_correct[i] / class_total[i] for i in range(10)]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#4CAF50' if a >= 70 else '#FF9800' if a >= 60 else '#F44336' for a in per_class_acc]
bars = ax.bar(classes, per_class_acc, color=colors, edgecolor='black', alpha=0.85)

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(y=np.mean(per_class_acc), color='navy', linestyle='--', linewidth=2, label=f'Average: {np.mean(per_class_acc):.1f}%')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy on CIFAR-10 Test Set', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

print("\nPer-class breakdown:")
for i, cls in enumerate(classes):
    print(f"  {cls:10s}: {per_class_acc[i]:.1f}%")

worst = classes[np.argmin(per_class_acc)]
best = classes[np.argmax(per_class_acc)]
print(f"\nBest:  {best} ({max(per_class_acc):.1f}%)")
print(f"Worst: {worst} ({min(per_class_acc):.1f}%)")
print("\nCat and deer are notoriously confused — they share similar shapes at 32x32 resolution.")

## Real World Problem: Accuracy vs Inference Speed

Google Photos classifies 1.2 billion photos per month. At that scale, a model that takes 100ms per image costs roughly 138 days of compute per month. At 10ms per image, it's 13.8 days.

The production problem is not "build the most accurate CNN." It's **"find the accuracy/speed tradeoff that meets your SLA."**

**What engineers actually do:**

1. **Use depthwise separable convolutions** (MobileNet): Factorises a standard conv into two cheaper operations. 8-9x fewer FLOPs, 1-2% accuracy drop. Used in Google Lens and mobile apps.

2. **Quantisation**: Convert float32 weights to int8. 4x smaller model, 2-4x faster inference, <1% accuracy drop. Standard for on-device ML.

3. **Pruning**: Remove weights below a threshold. A ResNet-50 can be pruned to 40% of its original size with ~2% accuracy drop.

4. **Knowledge distillation**: Train a smaller "student" model to mimic a larger "teacher" model. The student often outperforms a directly-trained model of the same size.

**Decision framework:**
- Server inference, batch processing: optimize for accuracy. ResNet, EfficientNet.
- Mobile/edge inference: optimize for size and speed. MobileNetV3, EfficientNet-Lite.
- Real-time video: optimize for latency. YOLO-family, MobileNet-SSD.

In [ ]:
# ============================================================
# Measure Inference Speed of Your CNN
# ============================================================
import time

model.eval()
dummy_batch = torch.randn(64, 3, 32, 32).to(device)

# Warm up
with torch.no_grad():
    for _ in range(5):
        _ = model(dummy_batch)

# Measure
timings = []
with torch.no_grad():
    for _ in range(20):
        start = time.perf_counter()
        _ = model(dummy_batch)
        timings.append(time.perf_counter() - start)

avg_ms = np.mean(timings) * 1000
per_image_ms = avg_ms / 64

print(f"Batch size:              64 images")
print(f"Avg batch time:          {avg_ms:.1f} ms")
print(f"Per-image inference:     {per_image_ms:.2f} ms")
print(f"Throughput:              {1000/per_image_ms:.0f} images/second")
print(f"\nAt this speed, to process 1.2B images (Google Photos monthly):")
hours = 1.2e9 * per_image_ms / 1000 / 3600
print(f"  Single process: {hours:,.0f} hours ({hours/24:.0f} days)")
print(f"  With 1000 parallel workers: {hours/1000:.1f} hours")
print("\nThis is why production teams use quantized, pruned models on specialized hardware (TPUs).")

## Interview Corner: MNC-Level Questions

---

**Q1: Why are CNNs better than fully connected networks for image data?**

*What they're testing:* Whether you understand the structural assumptions CNNs make about image data.

*Answer direction:* Three reasons. (1) Weight sharing: the same filter detects an edge anywhere in the image. A fully connected layer would need separate weights for every pixel position. (2) Local connectivity: a 3x3 conv only looks at a 3x3 region. Images have local structure — nearby pixels are correlated. (3) Translation invariance: max pooling makes the representation shift-invariant to small translations. Together, these reduce the parameter count by orders of magnitude (ResNet-50 has 25M params; a fully-connected equivalent would have billions) and encode the right inductive bias for spatial data.

---

**Q2: What does BatchNorm do in a CNN and why is it used after conv layers?**

*What they're testing:* Understanding of training stability techniques.

*Answer direction:* BatchNorm normalises the output of each layer to have mean≈0 and std≈1 across the batch dimension. This prevents the input distribution to the next layer from shifting during training (called internal covariate shift). Practically: it allows much higher learning rates, reduces sensitivity to weight initialisation, and acts as a mild regularizer. In CNNs it's applied per channel: each of the 32 channels in a 32-channel feature map gets its own learned scale (gamma) and shift (beta) parameters. Without BatchNorm you need very careful LR tuning and training often diverges.

---

**Q3: Explain the tradeoff between stride and max pooling for downsampling.**

*What they're testing:* Depth on CNN design choices.

*Answer direction:* Both halve spatial dimensions. Strided convolution is learnable: the downsampling filter is learned from data. Max pooling is a fixed operation: always take the maximum. In practice, architectures like ResNet use strided convolutions for downsampling because they preserve more information and are end-to-end differentiable with learnable kernels. Max pooling is faster but discards the average/minimum activations. The Striving for Simplicity paper (Springenberg et al., 2014) showed you can replace max pooling entirely with strided convolutions and get similar performance.

---

**Q4: A CNN deployed in production gives different results on the same image at different times. What could cause this?**

*What they're testing:* Production debugging mindset.

*Answer direction:* Most likely cause: the model is in training mode (`model.train()`) instead of eval mode (`model.eval()`). Dropout randomly zeros activations during training, and BatchNorm uses batch statistics instead of running statistics. Both are non-deterministic. Fix: always call `model.eval()` before inference. Also check GPU non-determinism: CUDA operations can be non-deterministic unless you set `torch.backends.cudnn.deterministic = True`. Confirm the preprocessing pipeline is also deterministic (no random augmentations at inference time).

---

**Q5: You train a CNN to 92% accuracy on training data but only 65% on test data. What's happening and how do you fix it?**

*What they're testing:* Overfitting diagnosis.

*Answer direction:* Overfitting: the model memorised the training set instead of learning generalisable patterns. Fixes in order of what to try first: (1) Data augmentation — random flips, crops, colour jitter, cutout. (2) Dropout — already in most CNN architectures. (3) Weight decay (L2 regularisation). (4) Reduce model capacity — fewer filters or fewer layers. (5) Get more training data or use transfer learning. In production, the 27% gap you described is severe and usually means the training set is too small or not representative of the test distribution. Check if train and test sets come from the same distribution before tuning the model.

## ML Spotlight

**EfficientNet (Google Brain, 2019) — The Most Efficient CNN Architecture**

EfficientNet introduced compound scaling: instead of scaling width, depth, or resolution independently, scale all three together using a fixed ratio. The result: EfficientNet-B7 achieves 84.3% top-1 accuracy on ImageNet with 8.4x fewer FLOPs and 6.8x fewer parameters than the best ResNet at the same accuracy.

It's currently the standard baseline for production image classification when you need accuracy at a controlled compute budget.

In PyTorch:
```python
from torchvision.models import efficientnet_b0
model = efficientnet_b0(pretrained=True)
```

Paper: [EfficientNet: Rethinking Model Scaling for CNNs](https://arxiv.org/abs/1905.11946)

GitHub: https://github.com/google/automl/tree/master/efficientdet

## Practice Exercise

**Task:** Modify the `CNNClassifier` above to add data augmentation and compare test accuracy.

1. Add random horizontal flip and random crop to the transform:
```python
transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
```
2. Retrain for 10 epochs. Compare test accuracy with and without augmentation.
3. Plot per-class accuracy for both runs. Which classes improve most?

**Expected result:** Augmentation should improve test accuracy by 3-6% in 10 epochs.

---

**What's Next**

Day 28: Transfer Learning — Why training from scratch is almost never necessary in production, and how to fine-tune a pretrained ResNet/MobileNet on your own dataset in under 50 lines of code.